# PubMed -> Vertex AI Preference Tuning Prep

Converts local tool-calling DPO JSONL into Gemini preference tuning JSONL format (`contents` + `completions`).

Source expected: `pubmed_oncologist_v2_tool_dpo_messages.jsonl`

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/home/spark/projects/training/pubmed')
SRC_FILE = PROJECT_ROOT / 'data' / 'training-data' / 'pubmed_oncologist_v2_tool_dpo_messages.jsonl'
OUT_DIR = PROJECT_ROOT / 'data' / 'training-data' / 'vertex_model_garden'
OUT_FILE = OUT_DIR / 'pubmed_vertex_preference.jsonl'

OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Source:', SRC_FILE)
print('Output:', OUT_FILE)

In [ ]:
import json
from typing import Any, Dict, List

def _try_json(s: str) -> Any:
    if not isinstance(s, str):
        return s
    s = s.strip()
    if not s:
        return ''
    try:
        return json.loads(s)
    except Exception:
        return s

def _common_prefix_len(a: List[Dict[str, Any]], b: List[Dict[str, Any]]) -> int:
    n = min(len(a), len(b))
    i = 0
    while i < n and a[i] == b[i]:
        i += 1
    return i

def _to_contents(messages: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out = []
    for m in messages:
        role = m.get('role')
        if role == 'system':
            # preference format uses optional system_instruction, not system inside contents
            continue
        if role == 'user':
            out.append({'role': 'user', 'parts': [{'text': m.get('content', '')}]})
        elif role == 'assistant':
            parts = []
            for tc in m.get('tool_calls', []):
                fn = tc.get('function', {})
                args = _try_json(fn.get('arguments', '{}'))
                if not isinstance(args, dict):
                    args = {'raw': args}
                parts.append({'functionCall': {'name': fn.get('name', ''), 'args': args}})
            text = m.get('content', '')
            if isinstance(text, str) and text.strip():
                parts.append({'text': text})
            if not parts:
                parts = [{'text': ''}]
            out.append({'role': 'model', 'parts': parts})
        elif role == 'tool':
            response_obj = _try_json(m.get('content', ''))
            if not isinstance(response_obj, dict):
                response_obj = {'output': response_obj}
            out.append({'parts': [{'functionResponse': {'name': m.get('name', ''), 'response': response_obj}}]})
    return out

def _to_single_completion(messages: List[Dict[str, Any]]) -> Dict[str, Any]:
    # Preference tuning requires exactly one model turn per completion.
    # If suffix has multiple turns, pick the first assistant turn when present
    # (this preserves the tool-call decision in tool-calling rows).
    assistant = None
    for m in messages:
        if m.get('role') == 'assistant':
            assistant = m
            break
    if assistant is None:
        return {'role': 'model', 'parts': [{'text': ''}]}

    parts = []
    for tc in assistant.get('tool_calls', []):
        fn = tc.get('function', {})
        args = _try_json(fn.get('arguments', '{}'))
        if not isinstance(args, dict):
            args = {'raw': args}
        parts.append({'functionCall': {'name': fn.get('name', ''), 'args': args}})

    text = assistant.get('content', '')
    if isinstance(text, str) and text.strip():
        parts.append({'text': text})

    if not parts:
        parts = [{'text': ''}]

    return {'role': 'model', 'parts': parts}

def convert_row(row: Dict[str, Any]) -> Dict[str, Any]:
    chosen = row.get('chosen', [])
    rejected = row.get('rejected', [])

    # Pull system instruction from first system message if present.
    system_text = ''
    if chosen and chosen[0].get('role') == 'system':
        system_text = chosen[0].get('content', '')

    # Shared prompt context.
    p = _common_prefix_len(chosen, rejected)
    prompt_msgs = chosen[:p] if p > 0 else chosen[:-1]

    # Preference completions.
    chosen_suffix = chosen[p:] if p > 0 else chosen[-1:]
    rejected_suffix = rejected[p:] if p > 0 else rejected[-1:]

    out = {
        'contents': _to_contents(prompt_msgs),
        'completions': [
            {'score': 1, 'completion': _to_single_completion(chosen_suffix)},
            {'score': 0, 'completion': _to_single_completion(rejected_suffix)},
        ],
    }

    if system_text:
        out['system_instruction'] = {'parts': [{'text': system_text}]}

    return out

In [ ]:
count = 0
with SRC_FILE.open('r', encoding='utf-8') as fin, OUT_FILE.open('w', encoding='utf-8') as fout:
    for line in fin:
        row = json.loads(line)
        out = convert_row(row)
        fout.write(json.dumps(out, ensure_ascii=False) + '\n')
        count += 1

print('Wrote rows:', count)
print('File size bytes:', OUT_FILE.stat().st_size)

In [ ]:
# Quick sanity check on first converted row
with OUT_FILE.open('r', encoding='utf-8') as f:
    first = json.loads(next(f))

print('Keys:', sorted(first.keys()))
print('Num prompt contents:', len(first.get('contents', [])))
print('Completions scores:', [c.get('score') for c in first.get('completions', [])])
print('Preferred completion role:', first.get('completions', [{}])[0].get('completion', {}).get('role'))

## Next Step in Google Cloud

1. Upload output JSONL to GCS.
2. In Model Garden / Agent Platform, launch preference tuning.
3. Use this JSONL as preference dataset.

Note: Gemini preference format expects one model turn per completion, so this notebook encodes each side as one model turn.